# Introduction

Ce notebook présente une solution pour la capture de la provenance des données dans les pipelines de préparation de données. La capture de la provenance est essentielle pour comprendre comment les enregistrements dans un jeu de données résultant d’une transformation ou d’une opération complexe sont reliés à leurs origines dans les jeux de données d’entrée.

### Objectifs

L’objectif principal est de développer une classe Python, appelée `TensorProv`, capable de :

1. Capturer la provenance des données pour trois types d’opérations courantes.
2. Représenter cette provenance à l’aide de tenseurs binaires creux (_sparse tensors_) pour assurer une gestion mémoire et un traitement efficace.
3. Tester et évaluer les performances des approches implémentées, en mesurant notamment les temps d’exécution et l’utilisation des ressources.

### Définition des Méthodes de Capture de Provenance

Deux approches principales sont explorées pour capturer la provenance des données :

1. **Méthode par hashing**  
   Cette méthode consiste à attribuer une clé unique à chaque enregistrement des jeux de données d’entrée en utilisant une fonction de hachage. Ces clés servent ensuite à identifier les correspondances entre les enregistrements des jeux de données de sortie et d’entrée. Bien que cette approche soit efficace pour tracer les liens entre les données, elle peut être coûteuse en termes de calcul pour les grands volumes de données.

2. **Méthode par ajout d’ID**  
   Une colonne supplémentaire contenant des identifiants uniques est ajoutée aux jeux de données d’entrée. Ces identifiants permettent de tracer directement chaque enregistrement des jeux d’entrée vers le jeu de données de sortie. Cette méthode est plus légère en termes de calcul mais requiert des modifications du schéma des données.


### Définition des Opérations Clés

#### **1. Data Transformation**
Les transformations de données consistent à modifier les valeurs spécifiques des attributs d’un jeu de données sans en changer la structure ni le nombre d’enregistrements. Ces transformations peuvent inclure des opérations telles que :
- La **normalisation**, qui ajuste les valeurs pour les ramener dans une échelle spécifique.
- La **discrétisation**, qui divise les valeurs continues en intervalles discrets.
- La **binarisation**, qui convertit les valeurs en données binaires (par exemple, 1 ou 0).

#### **2. Vertical Data Reduction**
Les opérations de réduction verticale consistent à supprimer des colonnes (ou attributs) d’un jeu de données. Cela peut inclure :
- **Feature Selection (sélection des variables)** : Identifier et conserver uniquement les colonnes pertinentes pour notre analyse.
- **Drop Columns (suppression des colonnes)** : Retirer des colonnes inutiles ou redondantes.

Le résultat est un jeu de données avec moins d’attributs mais contenant toujours le même nombre d’enregistrements.

#### **3. Horizontal Data Reduction**
La réduction horizontale vise à filtrer les lignes d’un jeu de données. Cela inclurs :
- **Filtrage** : Supprimer les lignes qui ne remplissent pas certaines conditions (par exemple, supprimer les lignes avec des valeurs manquantes ou aberrantes).
- **Undersampling (sous-échantillonnage)** : Réduire le nombre de lignes pour équilibrer les classes dans un problème de classification.
- **Row Deletion (suppression des lignes)** : Retirer explicitement certaines lignes.

Le résultat est un sous-ensemble des enregistrements d’origine.

#### **4. Vertical Data Augmentation**
L’augmentation verticale modifie le schéma d’un jeu de données sans en changer le nombre de lignes. Cela inclut :
- **Space Transformation** : Projeter les données dans un nouvel espace de caractéristiques (par exemple, utiliser une transformation PCA).
- **String Indexer** : Convertir les valeurs catégoriques en indices numériques.
- **One-Hot Encoding** : Créer des colonnes binaires pour représenter des catégories.

#### **5. Horizontal Data Augmentation**
L’augmentation horizontale génère de nouvelles lignes dans un jeu de données. Les opérations typiques incluent :
- **Oversampling (suréchantillonnage)** : Ajouter des duplications ou interpolations pour équilibrer les classes dans les données.
- **Instance Generation** : Générer artificiellement des enregistrements supplémentaires à partir des données existantes.

#### **6. Join (Fusion des Données)**
L’opération de jointure combine deux jeux de données en fonction d’une ou plusieurs colonnes communes. Cela inclut des types de jointures tels que :
- **Inner Join** : Conserver uniquement les enregistrements correspondants dans les deux jeux.
- **Left/Right Join** : Conserver tous les enregistrements d’un jeu de données et les correspondances dans l’autre.
- **Full Outer Join** : Conserver tous les enregistrements des deux jeux, en remplissant les valeurs manquantes avec `NULL`.

#### **7. Append (Concaténation)**
L’opération d’append ajoute les enregistrements d’un jeu de données à un autre. Contrairement à la jointure, les colonnes des deux jeux n’ont pas besoin d’être identiques :
- Les colonnes absentes sont complétées par des valeurs `NULL`.
- Les enregistrements du second jeu sont ajoutés à la fin du premier.

Cette opération est souvent utilisée pour assembler des ensembles de données qui partagent une structure similaire ou pour consolider des données provenant de différentes sources.


#Implémentation de la classe TensorProv

In [9]:
import pandas as pd
import numpy as np
import time
from scipy.sparse import coo_matrix
import hashlib

class TensorProv:
    """
    A class to capture provenance information for data transformations
    via two methods:

    1) Hashing-based approach
    2) Record-ID-based approach

    The resulting provenance object is a sparse binary tensor, indicating
    which input rows contributed to each output row.
    """

    def __init__(self, func, method='hash'):
        """
        Initialize the decorator.

        :param func: The original function (e.g. pd.DataFrame.query, pd.merge, etc.)
        :param method: 'hash' or 'id'. 'hash' uses hashing-based post-hoc approach,
                       'id' uses the record_id-based approach.
        """
        self.func = func
        self.method = method

    def __call__(self, *args, **kwargs):
        """
        When the decorated function is called, figure out which operation
        is being performed (by name) and call the appropriate 'decorate_*' method.
        """
        method_name = self.func.__name__.lower()

        if method_name in ["query", "filter"]:
            return self.decorate_filter(*args, **kwargs)

        elif method_name in ["merge", "join"]:
            return self.decorate_merge(*args, **kwargs)

        else:
            raise NotImplementedError(f"Decoration for '{method_name}' is not implemented.")


    # FILTER (horizontal data reduction)

    def decorate_filter(self, df, condition):
        """
        Decorates the filter operation.

        If method == 'hash', compute the provenance tensor by hashing.
        If method == 'id', assume df already has a 'record_id' column.
        """
        if self.method == 'hash':
            return self._filter_hash(df, condition)
        else:  # self.method == 'id'
            return self._filter_id(df, condition)

    def _filter_hash(self, df, condition):
        """
        Hashing-based approach to filter provenance:
          1. For each row in df, compute a hash.
          2. Apply filter, get filtered_df.
          3. For each row in filtered_df, match the hash to df to build a sparse matrix.
        """
        # Create a unique hash for each row in df
        input_hashes = df.apply(lambda row: self._row_hash(row), axis=1)

        # Evaluate the condition
        filtered_df = df.query(condition)

        # Create a unique hash for each row in filtered_df
        output_hashes = filtered_df.apply(lambda row: self._row_hash(row), axis=1)

        # Build provenance tensor in COO format
        row_indices = []
        col_indices = []

        hash_to_input_idx = {h: i for i, h in enumerate(input_hashes)}

        for j, h_out in enumerate(output_hashes):
            if h_out in hash_to_input_idx:
                i = hash_to_input_idx[h_out]
                row_indices.append(i)
                col_indices.append(j)

        data = np.ones(len(row_indices), dtype=np.int8)
        provenance_tensor = coo_matrix((data, (row_indices, col_indices)),
                                       shape=(len(df), len(filtered_df)))

        return filtered_df, provenance_tensor

    def _filter_id(self, df, condition):
        """
        ID-based approach for filter provenance:
          1. Assume df has a 'record_id' column.
          2. Filter the df.
          3. Build the provenance tensor using matching 'record_id's.
        """
        filtered_df = df.query(condition)

        input_ids = df['record_id'].values
        input_id_to_idx = {rid: i for i, rid in enumerate(input_ids)}

        filtered_ids = filtered_df['record_id'].values

        row_indices = []
        col_indices = []

        for j, rid in enumerate(filtered_ids):
            i = input_id_to_idx[rid]
            row_indices.append(i)
            col_indices.append(j)

        data = np.ones(len(row_indices), dtype=np.int8)
        provenance_tensor = coo_matrix((data, (row_indices, col_indices)),
                                       shape=(len(df), len(filtered_df)))

        return filtered_df, provenance_tensor

    # MERGE (data fusion)

    def decorate_merge(self, df1, df2, **kwargs):
        """
        Decorates the merge operation.

        If method == 'hash',  compute the provenance tensor by hashing.
        If method == 'id', assume df1 and df2 have 'record_id' columns.
        """
        if self.method == 'hash':
            return self._merge_hash(df1, df2, **kwargs)
        else:
            return self._merge_id(df1, df2, **kwargs)

    def _merge_hash(self, df1, df2, **kwargs):
        """
        Hashing-based approach for join provenance:
          1. Hash each row in both dataframes.
          2. Merge the dataframes.
          3. For each row in merged_df, try to find the corresponding row(s) in df1 and df2.
        """
        hashes_df1 = df1.apply(lambda row: self._row_hash(row), axis=1)
        hashes_df2 = df2.apply(lambda row: self._row_hash(row), axis=1)

        merged_df = self.func(df1, df2, **kwargs)

        total_input = len(df1) + len(df2)
        output_size = len(merged_df)

        row_indices = []
        col_indices = []

        hash_to_idx_df1 = {h: i for i, h in enumerate(hashes_df1)}
        hash_to_idx_df2 = {h: i + len(df1) for i, h in enumerate(hashes_df2)}

        merged_hashes = merged_df.apply(lambda row: self._row_hash(row), axis=1)

        for j, mh in enumerate(merged_hashes):
            if mh in hash_to_idx_df1:
                row_indices.append(hash_to_idx_df1[mh])
                col_indices.append(j)
            if mh in hash_to_idx_df2:
                row_indices.append(hash_to_idx_df2[mh])
                col_indices.append(j)

        data = np.ones(len(row_indices), dtype=np.int8)
        provenance_tensor = coo_matrix((data, (row_indices, col_indices)),
                                       shape=(total_input, output_size))

        return merged_df, provenance_tensor

    def _merge_id(self, df1, df2, **kwargs):
        """
        ID-based approach for join provenance:
          1. Assume df1, df2 each have a 'record_id' column.
          2. The join/merge carries these IDs forward in the result.
        """
        merged_df = self.func(df1, df2, **kwargs)

        total_input = len(df1) + len(df2)
        output_size = len(merged_df)

        row_indices = []
        col_indices = []

        input_ids_df1 = df1['record_id'].values
        input_ids_df2 = df2['record_id'].values

        df1_id_to_idx = {rid: i for i, rid in enumerate(input_ids_df1)}
        df2_id_to_idx = {rid: i + len(df1) for i, rid in enumerate(input_ids_df2)}

        # Pandas typically uses 'record_id_x' and 'record_id_y' after a merge
        if 'record_id_x' in merged_df.columns and 'record_id_y' in merged_df.columns:
            left_ids = merged_df['record_id_x'].values
            right_ids = merged_df['record_id_y'].values
        else:
            raise ValueError("Merged DataFrame does not contain the expected ID columns.")

        for j, (lx, rx) in enumerate(zip(left_ids, right_ids)):
            if pd.notnull(lx):
                row_indices.append(df1_id_to_idx[lx])
                col_indices.append(j)
            if pd.notnull(rx):
                row_indices.append(df2_id_to_idx[rx])
                col_indices.append(j)

        data = np.ones(len(row_indices), dtype=np.int8)
        provenance_tensor = coo_matrix((data, (row_indices, col_indices)),
                                       shape=(total_input, output_size))

        return merged_df, provenance_tensor   

    # HELPER for computing row hash

    @staticmethod
    def _row_hash(row):
        """
        Compute a hash for a row by converting row values into a string,
        then using SHA256 (or MD5, etc.).
        """
        items_str = "|".join(str(v) for v in row.values)
        return hashlib.sha256(items_str.encode('utf-8')).hexdigest()



In [10]:
# 2. OVERSAMPLING (Horizontal Data Augmentation)
#    - with hash-based approach
#    - with ID-based approach

def oversample_with_prov(df, times=2, method='hash'):
    """
    Oversampling example with provenance.
    - If method == 'hash', compute a hash for each row and replicate.
    - If method == 'id', replicate 'record_id' for each row.
    Return: (oversampled_df, provenance_tensor)
    """
    if method == 'hash':
        row_hashes = df.apply(lambda row: TensorProv._row_hash(row), axis=1)
        oversampled_df = pd.concat([df] * times, ignore_index=True)
        oversampled_hashes = pd.concat([row_hashes] * times, ignore_index=True)

        # Build sparse matrix
        input_size = len(df)
        output_size = len(oversampled_df)
        row_indices = []
        col_indices = []

        hash_to_input_idx = {h: i for i, h in enumerate(row_hashes)}

        for j, h in enumerate(oversampled_hashes):
            if h in hash_to_input_idx:
                i = hash_to_input_idx[h]
                row_indices.append(i)
                col_indices.append(j)

        data = np.ones(len(row_indices), dtype=np.int8)
        provenance_tensor = coo_matrix((data, (row_indices, col_indices)),
                                    shape=(input_size, output_size))
        return oversampled_df, provenance_tensor
    else:
        # ID-based approach
        oversampled_df = pd.concat([df] * times, ignore_index=True)

        # Build provenance
        input_ids = df['record_id'].values
        input_id_to_idx = {rid: i for i, rid in enumerate(input_ids)}

        repeated_ids = []
        for _ in range(times):
            repeated_ids.extend(input_ids)
        oversampled_df['record_id'] = repeated_ids  # keep same IDs or generate new ones

        row_indices = []
        col_indices = []
        for j, rid in enumerate(oversampled_df['record_id'].values):
            i = input_id_to_idx[rid]
            row_indices.append(i)
            col_indices.append(j)

        data = np.ones(len(row_indices), dtype=np.int8)
        provenance_tensor = coo_matrix((data, (row_indices, col_indices)),
                                    shape=(len(df), len(oversampled_df)))

        return oversampled_df, provenance_tensor


# 3. FEATURE SELECTION (Vertical Data Reduction)
#    - with hash-based approach
#    - with ID-based approach

def feature_select_with_prov(df, selected_cols, method='hash'):
    """
    Feature selection example:
    Vertical data reduction means columns are dropped.
    The row-level provenance typically remains the same (1:1).
    """
    if method == 'hash':
        selected_df = df[selected_cols].copy()
        input_size = len(df)
        output_size = len(selected_df)

        # Each output row depends on exactly one input row (same # of rows)
        row_indices = list(range(input_size))
        col_indices = list(range(output_size))

        data = np.ones(output_size, dtype=np.int8)
        provenance_tensor = coo_matrix((data, (row_indices, col_indices)),
                                    shape=(input_size, output_size))
        return selected_df, provenance_tensor

    else:  # ID-based approach
        selected_df = df[['record_id'] + selected_cols].copy()

        input_ids = df['record_id'].values
        input_id_to_idx = {rid: i for i, rid in enumerate(input_ids)}

        output_ids = selected_df['record_id'].values

        row_indices = []
        col_indices = []
        for j, rid in enumerate(output_ids):
            i = input_id_to_idx[rid]
            row_indices.append(i)
            col_indices.append(j)

        data = np.ones(len(row_indices), dtype=np.int8)
        provenance_tensor = coo_matrix((data, (row_indices, col_indices)),
                                    shape=(len(df), len(selected_df)))
        return selected_df, provenance_tensor


# 4. SPACE TRANSFORMATION (Vertical Data Augmentation)
#    - with hash-based approach
#    - with ID-based approach

def space_transform_with_prov(df, transform_func, method='hash'):
    """
    Space transformation (vertical augmentation):
    E.g., adding polynomial features or other transformations of existing columns.
    The number of rows stays the same, but new columns are added.
    """
    if method == 'hash':
        transformed_df = transform_func(df)

        # Row-level provenance: 1:1 for each row
        input_size = len(df)
        output_size = len(transformed_df)
        # For simplicity, assume the transform_func doesn't remove or add rows.

        row_indices = list(range(input_size))
        col_indices = list(range(output_size))

        data = np.ones(output_size, dtype=np.int8)
        provenance_tensor = coo_matrix((data, (row_indices, col_indices)),
                                    shape=(input_size, output_size))
        return transformed_df, provenance_tensor
    else:
        transformed_df = transform_func(df)
        # Re-attach record_id if not already done
        if 'record_id' not in transformed_df.columns:
            transformed_df['record_id'] = df['record_id'].values

        input_ids = df['record_id'].values
        input_id_to_idx = {rid: i for i, rid in enumerate(input_ids)}
        output_ids = transformed_df['record_id'].values

        row_indices = []
        col_indices = []
        for j, rid in enumerate(output_ids):
            i = input_id_to_idx[rid]
            row_indices.append(i)
            col_indices.append(j)

        data = np.ones(len(row_indices), dtype=np.int8)
        provenance_tensor = coo_matrix((data, (row_indices, col_indices)),
                                    shape=(len(df), len(transformed_df)))
        return transformed_df, provenance_tensor


In [11]:
# 5. TIME MEASUREMENT

def time_operation(operation, *args, **kwargs):
    """
    Measures the time to run the given operation.
    Expects the operation to return (df_result, provenance_tensor).
    Returns a tuple: (df_result, provenance_tensor, elapsed_time).
    """
    start_time = time.perf_counter()
    df_result, provenance_tensor = operation(*args, **kwargs)
    end_time = time.perf_counter()
    elapsed_time = end_time - start_time
    return df_result, provenance_tensor, elapsed_time

In [12]:
# Setup Example Data
df = pd.DataFrame({
    'A': [1, 2, 3, 4],
    'B': [5, 6, 7, 8]
})
df1 = pd.DataFrame({'key': [1, 2], 'value': ['A', 'B']})
df2 = pd.DataFrame({'key': [2, 3], 'value': ['C', 'D']})

# For ID-based approach, let's add a record_id column to each
df_id = df.copy()
df_id['record_id'] = df_id.index

df1_id = df1.copy()
df1_id['record_id'] = df1_id.index

df2_id = df2.copy()
df2_id['record_id'] = df2_id.index


# Operation Filtre

In [13]:
# 7.1 FILTER OPERATION

# a) Baseline filter
def baseline_filter(df, condition):
    return df.query(condition)

condition = 'A > 2'

# Baseline Filter
start = time.perf_counter()
filtered_df_baseline = baseline_filter(df, condition)
baseline_filter_time = time.perf_counter() - start

# Hash-based Filter
filter_with_prov_hash = TensorProv(pd.DataFrame.query, method='hash')
filtered_df_hash, filter_tensor_hash, filter_hash_time = time_operation(
    filter_with_prov_hash, df, condition
)

# ID-based Filter
filter_with_prov_id = TensorProv(pd.DataFrame.query, method='id')
filtered_df_id, filter_tensor_id, filter_id_time = time_operation(
    filter_with_prov_id, df_id, condition
)

print("=== FILTER OPERATION TIMING ===")
print(f"Baseline Filter (no provenance): {baseline_filter_time:.6f}s")
print(f"Hash-based Filter: {filter_hash_time:.6f}s")
print(f"ID-based Filter: {filter_id_time:.6f}s")


=== FILTER OPERATION TIMING ===
Baseline Filter (no provenance): 0.002706s
Hash-based Filter: 0.002561s
ID-based Filter: 0.003359s


# Operation - Merge

In [14]:
# 7.2 MERGE OPERATION

# b) Baseline merge
def baseline_merge(df1, df2, **kwargs):
    return pd.merge(df1, df2, **kwargs)


# Baseline Merge
start = time.perf_counter()
merged_df_baseline = baseline_merge(df1, df2, on='key', how='inner')
baseline_merge_time = time.perf_counter() - start

# Hash-based Merge
merge_with_prov_hash = TensorProv(pd.merge, method='hash')
merged_df_hash, merge_tensor_hash, merge_hash_time = time_operation(
    merge_with_prov_hash, df1, df2, on='key', how='inner'
)

# ID-based Merge
merge_with_prov_id = TensorProv(pd.merge, method='id')
merged_df_id, merge_tensor_id, merge_id_time = time_operation(
    merge_with_prov_id, df1_id, df2_id, on='key', how='inner'
)

print("\n=== MERGE OPERATION TIMING ===")
print(f"Baseline Merge (no provenance): {baseline_merge_time:.6f}s")
print(f"Hash-based Merge: {merge_hash_time:.6f}s")
print(f"ID-based Merge: {merge_id_time:.6f}s")



=== MERGE OPERATION TIMING ===
Baseline Merge (no provenance): 0.003483s
Hash-based Merge: 0.004225s
ID-based Merge: 0.003066s


# Operation - Owersampling

In [15]:
# 7.3 OVERSAMPLING

# c) Baseline oversampling
def baseline_oversample(df, times=2):
    return pd.concat([df]*times, ignore_index=True)

times = 2

# Baseline Oversample
start = time.perf_counter()
df_over_base = baseline_oversample(df, times=times)
over_base_time = time.perf_counter() - start

# Hash-based Oversample
df_over_hash, prov_over_hash, over_hash_time = time_operation(
    oversample_with_prov, df, times=times, method='hash'
)

# ID-based Oversample
df_over_id, prov_over_id, over_id_time = time_operation(
    oversample_with_prov, df_id, times=times, method='id'
)

print("\n=== OVERSAMPLING TIMING ===")
print(f"Baseline Oversample (no provenance): {over_base_time:.6f}s")
print(f"Hash-based Oversample: {over_hash_time:.6f}s")
print(f"ID-based Oversample: {over_id_time:.6f}s")



=== OVERSAMPLING TIMING ===
Baseline Oversample (no provenance): 0.000562s
Hash-based Oversample: 0.001111s
ID-based Oversample: 0.001377s


# Operation - Feature Selection

In [16]:
# 7.4 FEATURE SELECTION

# d) Baseline feature selection
def baseline_feature_selection(df, selected_cols):
    return df[selected_cols].copy()

selected_cols = ['A']

# Baseline Feature Selection
start = time.perf_counter()
df_fs_base = baseline_feature_selection(df, selected_cols)
fs_base_time = time.perf_counter() - start

# Hash-based Feature Selection
df_fs_hash, prov_fs_hash, fs_hash_time = time_operation(
    feature_select_with_prov, df, selected_cols=selected_cols, method='hash'
)

# ID-based Feature Selection
df_fs_id, prov_fs_id, fs_id_time = time_operation(
    feature_select_with_prov, df_id, selected_cols=selected_cols, method='id'
)

print("\n=== FEATURE SELECTION TIMING ===")
print(f"Baseline Feature Selection (no provenance): {fs_base_time:.6f}s")
print(f"Hash-based Feature Selection: {fs_hash_time:.6f}s")
print(f"ID-based Feature Selection: {fs_id_time:.6f}s")


=== FEATURE SELECTION TIMING ===
Baseline Feature Selection (no provenance): 0.000862s
Hash-based Feature Selection: 0.000618s
ID-based Feature Selection: 0.000870s


# Operation - Space Transformation

In [17]:
# 7.5 SPACE TRANSFORMATION

# e) Baseline space transformation
def baseline_space_transform(df, transform_func):
    return transform_func(df)

# Define a sample transform function that squares numeric columns
def sample_transform(df_in):
    df_out = df_in.copy()
    numeric_cols = df_out.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        df_out[col + "_squared"] = df_out[col] ** 2
    return df_out

# Baseline Space Transform
start = time.perf_counter()
df_st_base = baseline_space_transform(df, sample_transform)
st_base_time = time.perf_counter() - start

# Hash-based Space Transform
df_st_hash, prov_st_hash, st_hash_time = time_operation(
    space_transform_with_prov, df, transform_func=sample_transform, method='hash'
)

# ID-based Space Transform
df_st_id, prov_st_id, st_id_time = time_operation(
    space_transform_with_prov, df_id, transform_func=sample_transform, method='id'
)

print("\n=== SPACE TRANSFORMATION TIMING ===")
print(f"Baseline Space Transform (no provenance): {st_base_time:.6f}s")
print(f"Hash-based Space Transform: {st_hash_time:.6f}s")
print(f"ID-based Space Transform: {st_id_time:.6f}s")



=== SPACE TRANSFORMATION TIMING ===
Baseline Space Transform (no provenance): 0.001556s
Hash-based Space Transform: 0.001055s
ID-based Space Transform: 0.002219s
